In [12]:
!pip install transformers datasets evaluate seqeval -q

In [13]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate
import numpy as np

In [14]:
dataset = load_dataset("wikiann", "en")  # Or your chosen dataset

In [15]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [16]:
label_list = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for i,l in enumerate(label_list)}
num_labels = len(label_list)

In [17]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [20]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="no"
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),  # small demo
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    compute_metrics=compute_metrics
)

In [24]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    compute_metrics=compute_metrics,
    data_collator=data_collator  # <-- add this
)

In [26]:
# ================================
# 1️⃣1️⃣ Start Training
# ================================
trainer.train()

Step,Training Loss
10,1.650108
20,1.351360
30,1.354778
40,1.104874
50,1.174358
60,0.853100
70,0.799961
80,0.836039
90,0.750966
100,0.692677


TrainOutput(global_step=500, training_loss=0.5302865419387818, metrics={'train_runtime': 736.782, 'train_samples_per_second': 5.429, 'train_steps_per_second': 0.679, 'total_flos': 28221579325104.0, 'train_loss': 0.5302865419387818, 'epoch': 2.0})

In [28]:
# Example inference
sentence = "John works at Google in California"
tokens = tokenizer(sentence.split(), is_split_into_words=True, return_tensors="pt")
outputs = model(**tokens).logits
preds = np.argmax(outputs.detach().numpy(), axis=2)[0]
predicted_labels = [label_list[p] for p in preds]

print(list(zip(sentence.split(), predicted_labels)))

[('John', 'O'), ('works', 'B-PER'), ('at', 'I-PER'), ('Google', 'O'), ('in', 'B-LOC'), ('California', 'O')]
